# 📍 Redfin Distance Calculator
Upload your Redfin CSV and calculate the distance from each listing to your reference address.

## Step 1 — Install & Import

In [8]:
!pip install geopy pandas --quiet

import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import ssl
import certifi


print('✅ Ready')

✅ Ready


## Step 2 — ⚙️ Set Your Config

In [9]:
# ── Path to your downloaded Redfin CSV ──
CSV_PATH = "Desktop/Redfin Scraper"

# ── Your reference address (workplace, school, etc.) ──
REFERENCE_ADDRESS = "516 Ocean Rd, Santa Barbara, CA 93106"

# ── Output file name ──
OUTPUT_FILE = "redfin_with_distances_new.csv"

## Step 3 — Load CSV & Find Address Column

In [10]:
df = pd.read_csv("/Users/anugrhatamang/Desktop/Redfin Scraper/Santa Barbara/Redfin_listings_sb.csv")

print(f"✅ Loaded {len(df)} listings")
print(f"\nColumns found:\n{list(df.columns)}")
print(f"\nFirst few rows:")
display(df.head(3))

✅ Loaded 52 listings

Columns found:
['address', 'unit', 'bathrooms', 'bedrooms', 'description', 'price_from', 'distance_to_ucsb_miles']

First few rows:


,address,unit,bathrooms,bedrooms,description,price_from,distance_to_ucsb_miles
0,1330 Chapala St,NaN,2.0,2,Amenities\nIn-unit\nCeiling Fan\nRange\nRefrig...,4495.0,NaN
1,520 W Carrillo St,NaN,1.0,1,In-unit\nCarpet\nDisposal\nFurnished\nMicrowav...,2895.0,NaN
2,711 W Cota St,27,1.0,0,In-unit\nCeiling Fan\nRange\nRefrigerator\nSto...,2095.0,NaN


## Step 4 — Set Address Column
After running Step 3, look at the column names printed above and paste the address column name below.

In [11]:
ADDRESS_COLUMN = "address"  # keep this as whatever your address column is called

# Set any columns that don't exist in your CSV to None
CITY_COLUMN = None
STATE_COLUMN = None
ZIP_COLUMN = None

In [12]:
print(REFERENCE_ADDRESS)

516 Ocean Rd, Santa Barbara, CA 93106


In [13]:
df.columns.tolist()

['address',
 'unit',
 'bathrooms',
 'bedrooms',
 'description',
 'price_from',
 'distance_to_ucsb_miles']

In [14]:
df= df.drop(columns=['distance_to_ucsb_miles'])

## Step 5 — Geocode Reference Address

In [15]:
geolocator = Nominatim(user_agent="redfin_distance_calc")

# Geocode reference address
ref_location = geolocator.geocode(REFERENCE_ADDRESS, timeout=10)

if ref_location:
    ref_coords = (ref_location.latitude, ref_location.longitude)
    print(f"Reference coordinates: {ref_coords}")
else:
    raise ValueError("Reference address could not be geocoded")

Reference coordinates: (34.409938, -119.8534378)


## Step 6 — Calculate Distances
⏳ This may take a few minutes depending on how many listings you have (rate limited to 1 per second).

In [16]:
import requests
import time
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

geolocator = Nominatim(user_agent="your_app")
ORS_API_KEY = " "  # <-- paste your key here
# API Key Obtained by openrouteservice.org

def get_walking_distance(full_address, ref_coords):
    """Geocode address and return walking distance in miles via OpenRouteService."""
    if not ref_coords:
        return "N/A"
    try:
        time.sleep(1)  # Nominatim rate limit
        loc = geolocator.geocode(full_address, timeout=10)
        if not loc:
            return "N/A"

        # ORS expects [longitude, latitude]
        origin = [ref_coords[1], ref_coords[0]]
        destination = [loc.longitude, loc.latitude]

        response = requests.post(
            "https://api.openrouteservice.org/v2/directions/foot-walking",
            headers={"Authorization": ORS_API_KEY, "Content-Type": "application/json"},
            json={"coordinates": [origin, destination]}
        )

        if response.status_code == 200:
            data = response.json()
            meters = data["routes"][0]["summary"]["distance"]
            miles = round(meters / 1609.34, 1)
            return miles
        else:
            return "N/A"

    except Exception as e:
        print(f"Error for {full_address}: {e}")
        return "N/A"

distances = []
total = len(df)
for i, row in df.iterrows():
    addr = row["address"]
    dist = get_walking_distance(addr, ref_coords)
    distances.append(dist)
    print(f"[{i+1}/{total}] {addr}  →  {dist} mi")

df["distance_to_ucsb"] = distances
print(f"\n✅ Done! Walking distances calculated for {total} listings.")

[1/52] 1330 Chapala St  →  10.9 mi
[2/52] 520 W Carrillo St  →  10.7 mi
[3/52] 711 W Cota St  →  11.1 mi
[4/52] 711 W Cota St  →  11.1 mi
[5/52] 3999 Via Lucero  →  7.5 mi
[6/52] 3999 Via Lucero  →  7.5 mi
[7/52] 1035 Cliff Dr  →  11.7 mi
[8/52] 3940 Via Lucero  →  7.4 mi
[9/52] 3940 Via Lucero  →  7.4 mi
[10/52] 1 El Vedado Ln  →  9.3 mi
[11/52] 203 Ladera St  →  11.4 mi
[12/52] 203 Ladera St  →  11.4 mi
[13/52] 4099 Foothill Rd  →  7.8 mi
[14/52] 835 E Canon Perdido St  →  12.1 mi
[15/52] 100 Oceano Ave  →  11.8 mi
[16/52] 3885 State St  →  7.6 mi
[17/52] 3885 State St  →  7.6 mi
[18/52] 3885 State St  →  7.6 mi
[19/52] 3885 State St  →  7.6 mi
[20/52] 102 N Hope Ave  →  8.0 mi
[21/52] 102 N Hope Ave  →  8.0 mi
[22/52] 528 W Los Olivos St  →  9.7 mi
[23/52] 528 W Los Olivos St  →  9.7 mi
[24/52] 1210 Cacique St  →  13.6 mi
[25/52] 1825 Chapala St  →  10.3 mi
[26/52] 515 Red Rose Ln  →  11.3 mi
[27/52] 515 Red Rose Ln  →  11.3 mi
[28/52] 515 Red Rose Ln  →  11.3 mi
[29/52] 530 W Anapa

In [19]:
df.head

<bound method NDFrame.head of                    address                 unit  bathrooms  bedrooms  \
0          1330 Chapala St                  NaN        2.0         2   
1        520 W Carrillo St                  NaN        1.0         1   
2            711 W Cota St                   27        1.0         0   
3            711 W Cota St                    7        1.0         1   
4          3999 Via Lucero              Plan 1A        1.0         1   
5          3999 Via Lucero              Plan 1B        1.0         1   
6            1035 Cliff Dr                  NaN        1.0         2   
7          3940 Via Lucero                  NaN        1.0         0   
8          3940 Via Lucero                  NaN        1.0         1   
9           1 El Vedado Ln                  NaN        2.0         1   
10           203 Ladera St                  NaN        1.0         0   
11           203 Ladera St                  NaN        1.0         1   
12        4099 Foothill Rd        

## Step 7 — Preview & Save Results

In [21]:

import os

df_sorted['distance_to_ucsb'] = df_sorted['distance_to_ucsb'].round(2)
print(df_sorted.head())

path = '/Users/anugrhatamang/Desktop/Redfin Scraper/Santa Barbara/redfin_sb_uncleaned.csv'
df_sorted.to_csv(path, index=False)
print(f"✅ Saved to {path}")

NameError: name 'df_sorted' is not defined